# 04 — Evaluation & Threshold Selection

Train the CV-selected XGBoost pipeline, tune the decision threshold using validation data only, then evaluate the locked model and threshold exactly once on the untouched test set.

In [ ]:
import os
import sys
if os.getcwd().endswith("notebooks"):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, confusion_matrix

from src.data_loader import load_data
from src.evaluation import evaluate_at_threshold, evaluate_thresholds, find_best_f1_threshold
from src.model import build_xgboost
from src.preprocessing import split_data

In [ ]:
df = load_data("data/creditcard.csv")

X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    df,
    test_size=0.20,
    validation_size=0.20,
    random_state=42,
)

model = build_xgboost()
model.fit(X_train, y_train)

y_val_probability = model.predict_proba(X_val)[:, 1]

In [ ]:
default_metrics = evaluate_at_threshold(
    y_val,
    y_val_probability,
    threshold=0.50,
)
default_metrics

In [ ]:
best_f1 = find_best_f1_threshold(y_val, y_val_probability)
best_f1

In [ ]:
# Lock the threshold before touching test performance.
LOCKED_THRESHOLD = best_f1["threshold"]

y_test_probability = model.predict_proba(X_test)[:, 1]
test_metrics = evaluate_at_threshold(
    y_test,
    y_test_probability,
    threshold=LOCKED_THRESHOLD,
)

test_metrics

In [ ]:
Path("results").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

with open("results/final_test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

threshold_grid = np.linspace(0.01, 0.99, 99)
threshold_results = evaluate_thresholds(
    y_val,
    y_val_probability,
    threshold_grid,
)
threshold_results.to_csv("results/threshold_analysis.csv", index=False)

# Threshold trade-off
plt.figure(figsize=(9, 5.5))
plt.plot(threshold_results["threshold"], threshold_results["precision"], label="Precision")
plt.plot(threshold_results["threshold"], threshold_results["recall"], label="Recall")
plt.plot(threshold_results["threshold"], threshold_results["f1"], label="F1")
plt.axvline(LOCKED_THRESHOLD, linestyle="--", label=f"Selected = {LOCKED_THRESHOLD:.3f}")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Validation Threshold Analysis")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.savefig("figures/threshold_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

# Precision-recall curve
precision, recall, _ = precision_recall_curve(y_test, y_test_probability)
plt.figure(figsize=(7.5, 5.5))
plt.plot(recall, precision, label=f"XGBoost AUPRC = {test_metrics['auprc']:.3f}")
plt.axhline(y_test.mean(), linestyle="--", label=f"Fraud prevalence = {y_test.mean():.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve — Untouched Test Set")
plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.savefig("figures/precision_recall_curve.png", dpi=300, bbox_inches="tight")
plt.show()

# Confusion matrix
y_test_pred = (y_test_probability >= LOCKED_THRESHOLD).astype(int)
cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(6.3, 5.4))
image = ax.imshow(cm)
ax.set_xticks([0, 1], labels=["Legitimate", "Fraud"])
ax.set_yticks([0, 1], labels=["Legitimate", "Fraud"])
ax.set_xlabel("Predicted Class")
ax.set_ylabel("Actual Class")
ax.set_title("Final Test Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=14)

fig.colorbar(image)
plt.tight_layout()
plt.savefig("figures/confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()